# Generate Knowledge Graph

This notebook installs required packages, loads Neo4j credentials, imports cleaned preschool data from JSON, and builds a basic knowledge graph in Neo4j with Preschool, Town, and CareLevel nodes and their relationships.

In [1]:
!pip install neo4j
!pip install python-dotenv


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


This cell installs the required Neo4j and dotenv packages so the notebook can connect to Neo4j and securely load environment variables.

In [2]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

start_dir = Path.cwd().resolve()
repo_root = next(
    (path for path in (start_dir, *start_dir.parents) if (path / "SystemCode").is_dir()),
    None,
)
if repo_root is None:
    raise RuntimeError("Could not locate the KinderCompass repository root")
pipeline_dir = repo_root / "SystemCode/src/backend/pipeline"
if str(pipeline_dir) not in sys.path:
    sys.path.insert(0, str(pipeline_dir))

load_dotenv(repo_root / ".env")
print("URI loaded:", os.getenv("NEO4J_URI") is not None)
print("User loaded:", os.getenv("NEO4J_USERNAME"))
print("Password loaded:", os.getenv("NEO4J_PASSWORD") is not None)

URI loaded: True
User loaded: 0fd5a492
Password loaded: True


This cell loads Neo4j credentials from the .env file and prints whether each required connection variable was successfully discovered.

In [ ]:
# Guarded clear: set `confirm = True` to actually run the delete (non-destructive by default)
import os
from stage1.kg_client import get_driver
confirm = False  # Set to True only when intentionally rebuilding the entire graph.
if confirm:
    uri = os.getenv('NEO4J_URI')
    user = os.getenv('NEO4J_USERNAME')
    pwd = os.getenv('NEO4J_PASSWORD')
    if not uri or not user or not pwd:
        raise RuntimeError('Missing Neo4j credentials in environment')
    with get_driver() as driver:
        driver.verify_connectivity()
        with driver.session() as session:
            session.run('MATCH (n) DETACH DELETE n')
    print('Database cleared (executed)')
else:
    print('Clear skipped. Set confirm = True to execute the database clear.')

Database cleared (executed)


This cell clears the Neo4j database by deleting every node and relationship. Use this only when you want to rebuild the graph from scratch.

In [4]:
import json
import os
from pathlib import Path
from dotenv import load_dotenv
from stage1.kg_client import get_driver
import pandas as pd
import time

# 1. Load environment variables from .env
load_dotenv(repo_root / ".env")

# 2. Load the cleaned master JSON file
json_path = repo_root / "SystemCode/data/processed/kindercompass_master.json"

with open(json_path, "r", encoding="utf-8") as f:
    preschool_data = json.load(f)

# 3. Define optimized Cypher queries
# Query 1: Creates the core Preschool and its Town relationship
create_preschool_query = """
MERGE (p:Preschool {school_id: $school_id})
SET p.centre_code = $centre_code,
    p.tp_code = $tp_code,
    p.identifier_type = $identifier_type,
    p.name = $name,
    p.postal_code = $postal_code,
    p.base_fee = $base_fee,
    p.operator_scheme = $operator_scheme,
    p.care_levels = $care_levels,
    p.philosophy = $philosophy,
    p.pedagogy = $pedagogy,
    p.second_languages_offered = $second_languages_offered,
    p.spark_certified = $spark_certified,
    p.service_model = $service_model,
    p.food_offered = $food_offered,
    p.weekday_full_day = $weekday_full_day,
    p.provision_of_transport = $provision_of_transport,
    p.last_updated = $last_updated
WITH p
OPTIONAL MATCH (p)-[old_location:LOCATED_IN]->(:Town)
DELETE old_location
WITH p
FOREACH (_ IN CASE WHEN $town IS NULL THEN [] ELSE [1] END |
    MERGE (t:Town {name: $town})
    MERGE (p)-[:LOCATED_IN]->(t)
)
"""

# Query 2: Unwinds all care levels in a single operation
add_care_levels_query = """
MATCH (p:Preschool {school_id: $school_id})
UNWIND $levels as level_name
WITH p, level_name WHERE level_name IS NOT NULL
MERGE (c:CareLevel {name: level_name})
MERGE (p)-[:SERVES_LEVEL]->(c)
"""

start_time = time.perf_counter()

#1. verify_connectivity() tries Bolt
#2. Neo4j driver prints “Unable to retrieve routing information”
#3. The client catches ServiceUnavailable
#4. It verifies the HTTPS Query API
#5. Transport changes to HTTPS
#6. Processing continues through HTTPS

# 4. Initialize driver connection and run ingestion pipeline
with get_driver() as driver:
    driver.verify_connectivity()
    
    with driver.session() as session:
        total_schools = len(preschool_data)
        
        for i, school in enumerate(preschool_data):
            if i % 50 == 0:
                elapsed = time.perf_counter() - start_time
                print(f"Processing school {i} of {total_schools}... (elapsed: {elapsed:.2f}s)")
            
            # Extract clean scalar properties
            current_school_id = school.get("school_id")
            current_centre_code = school.get("centre_code")
            current_tp_code = school.get("tp_code")
            current_name = school.get("centre_name_x")
            current_postal_code = school.get("postal_code")
            current_town = school.get("town")
            current_base_fee = float(school["base_fee"]) if pd.notnull(school.get("base_fee")) else None
            current_operator_scheme = school.get("operator_scheme")
            current_care_levels = school.get("care_levels") or []
            current_philosophy = school.get("philosophy")
            current_pedagogy = school.get("pedagogy")
            
            # Extract all levels into a clean list for the UNWIND query
            levels_list = []
            if school.get("services_menu") is not None:
                for service in school["services_menu"]:
                    level = service.get("levels_offered")
                    if level:
                        levels_list.append(level)

            # --- SOLUTION 1 FIXED PARADIGM ---
            # Use the wrapper's own session.run() directly without transaction context managers.
            # 1. Create/update the core preschool and location node
            session.run(
                create_preschool_query,
                school_id=current_school_id,
                centre_code=current_centre_code,
                tp_code=current_tp_code,
                identifier_type=school.get("identifier_type"),
                name=current_name,
                postal_code=current_postal_code,
                town=current_town,
                base_fee=current_base_fee,
                operator_scheme=current_operator_scheme,
                care_levels=current_care_levels,
                philosophy=current_philosophy,
                pedagogy=current_pedagogy,
                second_languages_offered=school.get("second_languages_offered"),
                spark_certified=school.get("spark_certified"),
                service_model=school.get("service_model"),
                food_offered=school.get("food_offered"),
                weekday_full_day=school.get("weekday_full_day"),
                provision_of_transport=school.get("provision_of_transport"),
                last_updated=school.get("last_updated")
            )
            
            # 2. Wire up care levels conditionally via list parameter mapping
            if levels_list:
                session.run(add_care_levels_query, school_id=current_school_id, levels=levels_list)

        total_elapsed = time.perf_counter() - start_time
        print(f"All preschools, towns, and care levels imported successfully! 🎉 (total elapsed: {total_elapsed:.2f}s)")


Processing school 0 of 1816... (elapsed: 0.10s)
Processing school 50 of 1816... (elapsed: 1.60s)
Processing school 100 of 1816... (elapsed: 3.15s)
Processing school 150 of 1816... (elapsed: 4.68s)
Processing school 200 of 1816... (elapsed: 6.13s)
Processing school 250 of 1816... (elapsed: 7.41s)
Processing school 300 of 1816... (elapsed: 8.59s)
Processing school 350 of 1816... (elapsed: 9.88s)
Processing school 400 of 1816... (elapsed: 11.21s)
Processing school 450 of 1816... (elapsed: 12.30s)
Processing school 500 of 1816... (elapsed: 13.38s)
Processing school 550 of 1816... (elapsed: 14.65s)
Processing school 600 of 1816... (elapsed: 16.01s)
Processing school 650 of 1816... (elapsed: 17.28s)
Processing school 700 of 1816... (elapsed: 18.43s)
Processing school 750 of 1816... (elapsed: 19.75s)
Processing school 800 of 1816... (elapsed: 21.02s)
Processing school 850 of 1816... (elapsed: 22.30s)
Processing school 900 of 1816... (elapsed: 23.50s)
Processing school 950 of 1816... (elapsed: